# 第 2 章：处理文本数据

## 流水线：原始文本 → 分词 → token ID → 滑动窗口 → DataLoader 批次

本章目标：把人类可读的文字，转化成 GPT 能消费的训练样本 `(input_batch, target_batch)`。

**五个步骤：**
1. 读取语料
2. 分词（简单分词 → BPE）
3. token ↔ 整数 ID 映射
4. 滑动窗口生成 (input, target) 对
5. 封装成 PyTorch DataLoader

> 📌 本章核心组件已提取到 `src/gpt/data.py`，供后续所有章节复用。

## 1. 读取预训练语料 "The Verdict"

用原书提供的短篇故事作为预训练 demo 语料（仅约 2 万字符，适合快速实验）。

In [1]:
# 读取语料（notebook 从 ch02-text-data/ 目录运行，用相对路径 ../data）
with open("../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"语料总字符数: {len(raw_text)}")
print(f"前 100 个字符:\n{raw_text[:100]}")

语料总字符数: 20479
前 100 个字符:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


## 2. 简单分词（教学过渡）

在引入真正的 BPE 之前，先用正则做一个最朴素的分词：按空白和标点切分。
**这是过渡方案**，目的是建立直觉——它的缺陷会解释为什么需要 BPE。

In [2]:
import re

# 按空白和常见标点切分（注意：括号里的捕获组会保留标点本身）
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(f"切分后 token 数: {len(preprocessed)}")
print(f"前 30 个 token: {preprocessed[:30]}")

切分后 token 数: 4649
前 30 个 token: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 3. 构建词表与 字符串↔整数 映射

模型不认识字符串，只认数字。我们需要：
- **词表（vocabulary）**：所有 token 的去重排序列表
- **str_to_int**：token → 整数 ID
- **int_to_str**：整数 ID → token
- 特殊 token：`<|endoftext|>`（文本结束）、`<|unk|>`（未知词）

In [3]:
# 去重 + 排序，得到词表
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])  # 特殊 token：文本结束 / 未知词

vocab_size = len(all_tokens)
print(f"词表大小: {vocab_size}")

# 字符串 → 整数 ID
str_to_int = {token: i for i, token in enumerate(all_tokens)}
# 整数 ID → 字符串
int_to_str = {i: token for i, token in enumerate(all_tokens)}

def simple_tokenizer(text):
    """简单分词器：文本 → token ID 列表"""
    tokens = re.split(r'([,.?_!"()\']|--|\s)', text)
    tokens = [t.strip() for t in tokens if t.strip()]
    return [str_to_int.get(t, str_to_int["<|unk|>"]) for t in tokens]

# 验证
ids = simple_tokenizer("Hello, world!")
print(f"token IDs: {ids}")

词表大小: 1161
token IDs: [1160, 5, 1160, 0]


## 4. 引入 BPE 分词器（tiktoken）

### 简单分词的问题
- 词表会随语料无限膨胀
- 遇到训练时没见过的词（新人名、拼写错误），只能映射成 `<|unk|>`，丢失信息

### BPE（Byte Pair Encoding）如何解决
BPE 把**未知词拆成已知子词**。例如 "unhappiness" 可能被拆成 `["un", "happiness"]` 或 `["un", "h", "app", "iness"]`。
这样词表固定（如 GPT-2 的 50257），却能泛化到几乎任何词。

> 本书直接用 OpenAI 的 BPE 实现 **tiktoken**（GPT-2/GPT-3 同款分词器）。
> 想深入 BPE 算法本身，见官方 `ch02/05_bpe-from-scratch/`（roadmap 中的 bonus）。

In [4]:
import tiktoken

# 加载 GPT-2 的 BPE 分词器（词表大小 50257）
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world! Is this-- a test?"
ids = tokenizer.encode(text)         # 文本 → token ID
decoded = tokenizer.decode(ids)      # token ID → 文本

print(f"BPE 编码: {ids}")
print(f"解码还原: {decoded}")
print(f"GPT-2 词表大小: {tokenizer.n_vocab}")

BPE 编码: [15496, 11, 995, 0, 1148, 428, 438, 257, 1332, 30]
解码还原: Hello, world! Is this-- a test?
GPT-2 词表大小: 50257


## 5. 滑动窗口生成 (input, target) 训练对

LLM 是**自回归**模型：根据前 N 个 token 预测第 N+1 个 token。
用滑动窗口把长 token 序列切成无数 (input, target) 训练对：

```
input  = token[i : i+max_len]
target = token[i+1 : i+max_len+1]   ← target 比 input 右移一位
```

> 这个"右移一位"是自回归语言模型的核心——模型学的是"看到上文，预测下一个词"。

In [5]:
import torch

# 先把整篇语料编码成 token ID
with open("../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(f"整篇语料 token 数: {len(enc_text)}")

C:\Users\guobi\AppData\Roaming\Python\Python314\site-packages\torch\cuda\__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


整篇语料 token 数: 5145


In [6]:
context_size = 4         # 上下文长度（demo 用小值，真实 GPT 是 256/1024）
stride = 1               # 滑动步长

# 切出前几对 (input, target) 演示
for i in range(0, 8):    # 只演示前 8 对
    input_chunk = enc_text[i : i + context_size]
    target_chunk = enc_text[i + 1 : i + context_size + 1]
    print(f"--- 第 {i} 对 ---")
    print(f"input : {input_chunk}  → {tokenizer.decode(input_chunk)}")
    print(f"target: {target_chunk}  → {tokenizer.decode(target_chunk)}")

--- 第 0 对 ---
input : [40, 367, 2885, 1464]  → I HAD always
target: [367, 2885, 1464, 1807]  →  HAD always thought
--- 第 1 对 ---
input : [367, 2885, 1464, 1807]  →  HAD always thought
target: [2885, 1464, 1807, 3619]  → AD always thought Jack
--- 第 2 对 ---
input : [2885, 1464, 1807, 3619]  → AD always thought Jack
target: [1464, 1807, 3619, 402]  →  always thought Jack G
--- 第 3 对 ---
input : [1464, 1807, 3619, 402]  →  always thought Jack G
target: [1807, 3619, 402, 271]  →  thought Jack Gis
--- 第 4 对 ---
input : [1807, 3619, 402, 271]  →  thought Jack Gis
target: [3619, 402, 271, 10899]  →  Jack Gisburn
--- 第 5 对 ---
input : [3619, 402, 271, 10899]  →  Jack Gisburn
target: [402, 271, 10899, 2138]  →  Gisburn rather
--- 第 6 对 ---
input : [402, 271, 10899, 2138]  →  Gisburn rather
target: [271, 10899, 2138, 257]  → isburn rather a
--- 第 7 对 ---
input : [271, 10899, 2138, 257]  → isburn rather a
target: [10899, 2138, 257, 7026]  → burn rather a cheap


## 6. 封装成 PyTorch DataLoader

把上述流程封装成两个可复用组件（已放在 `src/gpt/data.py`）：

- **`GPTDatasetV1`**：对编码后的序列施加滑动窗口，每个样本是 `(input_chunk, target_chunk)`
- **`create_dataloader_v1`**：把 Dataset 包装成批处理的 DataLoader

> 注意：真实训练中 `max_length` 通常设为 256，`stride` 等于 `max_length`（避免样本重叠），
> `batch_size` 设为 GPU 能容纳的大小。这里用小值仅为演示。

In [7]:
# 从 src/gpt 复用本章核心组件
import sys; sys.path.insert(0, "..")
from src.gpt.data import create_dataloader_v1

dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

# 取前 2 个批次展示
for batch_idx, (x, y) in enumerate(dataloader):
    if batch_idx >= 2:
        break
    print(f"批次 {batch_idx}: x={x.shape}, y={y.shape}")

print()
print("关键性质验证：target 应为 input 右移一位")
first_x, first_y = next(iter(dataloader))
assert torch.equal(first_x[:, 1:], first_y[:, :-1]), "target 应为 input 右移一位"
print("✅ 验证通过：每个样本内 target == input 右移一位")

批次 0: x=torch.Size([8, 4]), y=torch.Size([8, 4])
批次 1: x=torch.Size([8, 4]), y=torch.Size([8, 4])

关键性质验证：target 应为 input 右移一位
✅ 验证通过：每个样本内 target == input 右移一位


## 小结

本章完成了一条完整的数据流水线：

| 步骤 | 输入 | 输出 | 工具 |
|------|------|------|------|
| 1. 读取 | — | 原始文本 `str` | `open()` |
| 2. 分词 | 文本 | token ID 列表 | `tiktoken` (BPE) |
| 3. 滑动窗口 | token 序列 | `(input, target)` 对 | `GPTDatasetV1` |
| 4. 批处理 | Dataset | `(input_batch, target_batch)` | `DataLoader` |

**下一步（第 3 章）**：这些 token ID 还只是离散整数。要让模型理解它们的语义，
需要把它们变成**稠密向量**——这就是**词嵌入（embedding）**。
注意力机制将作用在这些 embedding 之上。

> 📎 本章代码整理版见 `solution.py`，复用模块见 `src/gpt/data.py`。